# Construct pydantic model from text input

In [3]:
from pydantic_ai import Agent
from utils.helpers import nvidia_model

ai_model = nvidia_model()

py_agent = Agent(model=ai_model)

first_result = await py_agent.run("Give me an IT employee working in sweden, keep it short")

first_result

                 pydantic-ai v2.63.0 • Python 3.13.7

      / \        agent: py_agent • model: openai:nvidia/nemotron-3.5-super-vl-120b-a12b • tools: 0
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


AgentRunResult(output='Emma Andersson, 32, is a backend developer at a SaaS startup in Stockholm. She works with Python, Go, and AWS in a flat, English-speaking team. Her routine includes daily stand-ups, fika breaks, and flexible hours, reflecting Sweden’s strong work-life balance and four-week summer holidays. Fluent in Swedish and English.')

In [5]:
print(first_result.output)

Emma Andersson, 32, is a backend developer at a SaaS startup in Stockholm. She works with Python, Go, and AWS in a flat, English-speaking team. Her routine includes daily stand-ups, fika breaks, and flexible hours, reflecting Sweden’s strong work-life balance and four-week summer holidays. Fluent in Swedish and English.


In [6]:
from pydantic import BaseModel, Field

class EmployeeModel(BaseModel):
    name: str
    age: int
    salary: int = Field(gt=30_000, lt=50_000)
    position: str

structured_result = await py_agent.run("Give me an IT employee working in sweden, keep it short", output_type=EmployeeModel)

structured_result

AgentRunResult(output=EmployeeModel(name='Erik Andersson', age=32, salary=45000, position='Software Engineer'))

In [7]:
structured_result.output

EmployeeModel(name='Erik Andersson', age=32, salary=45000, position='Software Engineer')

In [8]:
employee = structured_result.output

employee

EmployeeModel(name='Erik Andersson', age=32, salary=45000, position='Software Engineer')

In [10]:
employee.name

'Erik Andersson'

In [11]:
employee.model_dump()

{'name': 'Erik Andersson',
 'age': 32,
 'salary': 45000,
 'position': 'Software Engineer'}

In [13]:
print(employee.model_dump_json(indent=2))

{
  "name": "Erik Andersson",
  "age": 32,
  "salary": 45000,
  "position": "Software Engineer"
}


In [14]:
result_more_employees = await py_agent.run("Give me 10 in AI and data engineering fields, roles can vary, but salary must be between 30000 and 50000", output_type=list[EmployeeModel])

employees = result_more_employees.output

employees

[EmployeeModel(name='Alice Smith', age=28, salary=42000, position='Machine Learning Engineer'),
 EmployeeModel(name='Bob Johnson', age=34, salary=38000, position='Data Engineer'),
 EmployeeModel(name='Carol Lee', age=31, salary=45000, position='AI Research Scientist'),
 EmployeeModel(name='David Kim', age=29, salary=41000, position='Data Scientist'),
 EmployeeModel(name='Eva Martinez', age=33, salary=37000, position='Big Data Engineer'),
 EmployeeModel(name='Frank Wang', age=27, salary=43000, position='NLP Engineer'),
 EmployeeModel(name='Grace Okafor', age=30, salary=46000, position='AI Solutions Architect'),
 EmployeeModel(name='Henry Chen', age=35, salary=39000, position='Data Analyst'),
 EmployeeModel(name='Isabella Rossi', age=26, salary=44000, position='Computer Vision Engineer'),
 EmployeeModel(name='Jack Thompson', age=32, salary=36000, position='MLOps Engineer')]

In [15]:
len(employees)

10

In [16]:
for employ in employees:
    print(f"{employ.name} is {employ.age} years old and works as a {employ.position} and have salary of {employ.salary} kr")

Alice Smith is 28 years old and works as a Machine Learning Engineer and have salary of 42000 kr
Bob Johnson is 34 years old and works as a Data Engineer and have salary of 38000 kr
Carol Lee is 31 years old and works as a AI Research Scientist and have salary of 45000 kr
David Kim is 29 years old and works as a Data Scientist and have salary of 41000 kr
Eva Martinez is 33 years old and works as a Big Data Engineer and have salary of 37000 kr
Frank Wang is 27 years old and works as a NLP Engineer and have salary of 43000 kr
Grace Okafor is 30 years old and works as a AI Solutions Architect and have salary of 46000 kr
Henry Chen is 35 years old and works as a Data Analyst and have salary of 39000 kr
Isabella Rossi is 26 years old and works as a Computer Vision Engineer and have salary of 44000 kr
Jack Thompson is 32 years old and works as a MLOps Engineer and have salary of 36000 kr


## CV model

In [20]:
class ExperienceModel(BaseModel):
    title: str
    company: str
    description: str
    start_year: int
    end_year: int

class EducationModel(BaseModel):
    title: str
    education_area: str
    school: str
    description: str
    start_year: int
    end_year: int

class CvModel(BaseModel):
    name: str
    age: int
    experiences: list[ExperienceModel]
    education: list[EducationModel]

base_model_result = await py_agent.run("Create a swedish person applying for a data engineering position", output_type=CvModel)

resume = base_model_result.output

resume

CvModel(name='Erik Andersson', age=31, experiences=[ExperienceModel(title='Senior Data Engineer', company='Spotify AB', description='Designed and maintained scalable data pipelines on GCP using Apache Beam, BigQuery, and Dataflow. Collaborated with analytics and product teams to enable data-driven decision making across the music streaming platform.', start_year=2020, end_year=2024), ExperienceModel(title='Data Engineer', company='Klarna Bank AB', description='Built ETL pipelines and data warehouses using Python, SQL, and Apache Airflow. Supported machine learning feature stores for credit risk scoring and fraud detection models.', start_year=2017, end_year=2020), ExperienceModel(title='Software Developer', company='Ericsson', description='Developed backend services for telecom data processing using Java and Kafka. Worked on real-time streaming solutions for network performance monitoring.', start_year=2015, end_year=2017)], education=[EducationModel(title='Master of Science in Compute

In [22]:
resume.name, resume.age

('Erik Andersson', 31)

In [23]:
resume.experiences[0].title

'Senior Data Engineer'

## Optional postprocessing -> load into duckdb and unnest

In [24]:
import dlt

db_pipeline = dlt.pipeline(
    pipeline_name="resume_json_duckdb",
    destination=dlt.destinations.duckdb("cv.duckdb"),
    dataset_name="staging"
)

info = db_pipeline.run(data=[resume.model_dump()], loader_file_format="jsonl", table_name="cv_entries")

print(info)

Pipeline resume_json_duckdb load step finished in 0.34 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\user\Documents\github\AI_01_DE25_Rikard_Oledal\03_pydantic_model\cv.duckdb location to store data
Load package 1793612158.2872896 is LOADED and contains no failed jobs


In [27]:
import duckdb

with duckdb.connect("cv.duckdb") as conn:
    desc = conn.sql("desc").df()
    cv_entries  = conn.sql("from staging.cv_entries").df()
    educations  = conn.sql("from staging.cv_entries__education").df()
    experiences = conn.sql("from staging.cv_entries__experiences").df()

desc

,database,schema,name,column_names,column_types,temporary
0,cv,staging,_dlt_loads,"[load_id, schema_name, status, inserted_at, sc...","[VARCHAR, VARCHAR, BIGINT, TIMESTAMP WITH TIME...",False
1,cv,staging,_dlt_pipeline_state,"[version, engine_version, pipeline_name, state...","[BIGINT, BIGINT, VARCHAR, VARCHAR, TIMESTAMP W...",False
2,cv,staging,_dlt_version,"[version, engine_version, inserted_at, schema_...","[BIGINT, BIGINT, TIMESTAMP WITH TIME ZONE, VAR...",False
3,cv,staging,cv_entries,"[name, age, _dlt_load_id, _dlt_id]","[VARCHAR, BIGINT, VARCHAR, VARCHAR]",False
4,cv,staging,cv_entries__education,"[title, education_area, school, description, s...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, BIGINT, B...",False
5,cv,staging,cv_entries__experiences,"[title, company, description, start_year, end_...","[VARCHAR, VARCHAR, VARCHAR, BIGINT, BIGINT, VA...",False


In [28]:
cv_entries

,name,age,_dlt_load_id,_dlt_id
0,Erik Andersson,31,1793612158.2872896,uLQXwvQMFmhNpg


In [29]:
educations

,title,education_area,school,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,Master of Science in Computer Science and Engi...,Data Engineering,KTH Royal Institute of Technology,"Specialized in big data systems, distributed c...",2013,2016,uLQXwvQMFmhNpg,0,4sVWpehVNocc8Q
1,Bachelor of Science in Computer Science,Computer Science,Linköping University,"Foundation in programming, algorithms, databas...",2010,2013,uLQXwvQMFmhNpg,1,7c0dZKPSUVyG/Q


In [30]:
experiences

,title,company,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,Senior Data Engineer,Spotify AB,Designed and maintained scalable data pipeline...,2020,2024,uLQXwvQMFmhNpg,0,FZKrdLdnOL0nnA
1,Data Engineer,Klarna Bank AB,Built ETL pipelines and data warehouses using ...,2017,2020,uLQXwvQMFmhNpg,1,WSYd6JHhuDX6XA
2,Software Developer,Ericsson,Developed backend services for telecom data pr...,2015,2017,uLQXwvQMFmhNpg,2,MfHwwxwKcZG7Qw


In [33]:
duckdb.sql("""
    SELECT
        cv.name,
        cv.age,
        ex.company,
        ex.description AS experience_description,
        ex.start_year AS experience_start_year,
        ex.end_year AS experience_end_year,
        ed.title,
        ed.education_area,
        ed.school,
        ed.start_year AS education_start_year,
        ed.end_year AS education_end_year
    FROM cv_entries cv
    LEFT JOIN educations ed ON cv._dlt_id = ed._dlt_parent_id
    LEFT JOIN experiences ex ON cv._dlt_id = ex._dlt_parent_id
""").df()

,name,age,company,experience_description,experience_start_year,experience_end_year,title,education_area,school,education_start_year,education_end_year
0,Erik Andersson,31,Spotify AB,Designed and maintained scalable data pipeline...,2020,2024,Bachelor of Science in Computer Science,Computer Science,Linköping University,2010,2013
1,Erik Andersson,31,Klarna Bank AB,Built ETL pipelines and data warehouses using ...,2017,2020,Bachelor of Science in Computer Science,Computer Science,Linköping University,2010,2013
2,Erik Andersson,31,Ericsson,Developed backend services for telecom data pr...,2015,2017,Bachelor of Science in Computer Science,Computer Science,Linköping University,2010,2013
3,Erik Andersson,31,Spotify AB,Designed and maintained scalable data pipeline...,2020,2024,Master of Science in Computer Science and Engi...,Data Engineering,KTH Royal Institute of Technology,2013,2016
4,Erik Andersson,31,Klarna Bank AB,Built ETL pipelines and data warehouses using ...,2017,2020,Master of Science in Computer Science and Engi...,Data Engineering,KTH Royal Institute of Technology,2013,2016
5,Erik Andersson,31,Ericsson,Developed backend services for telecom data pr...,2015,2017,Master of Science in Computer Science and Engi...,Data Engineering,KTH Royal Institute of Technology,2013,2016
